# LAB S07: Automatización, despliegue y monitoreo

Cada sección usa el archivo de `mi_pipeline` que se escribe en su paso de la guía; el cuaderno solo importa y ejecuta. Hoy el pipeline de la sesión 6 se **empaqueta** (una rueda que se instala en un entorno limpio), se **instrumenta** (un registro sin datos personales y un tablero con alertas), se **prueba** contra un golden set con una versión etiquetada y **arranca seguro** (secretos fuera del código y fallo temprano). No se usan modelos de lenguaje ni contenedores en la VM. Los documentos son ficticios. Ningún documento sale de la VM.

## 0. Dónde están los datos de práctica

In [ ]:
# Celda estándar de localización de datos (no se modifica)
import os
from pathlib import Path


def localizar_carpeta(nombre="02_Datos_Practica", max_niveles=6, obligatorio=True):
    """Busca `nombre` en la carpeta actual y en sus carpetas superiores."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        candidata = carpeta / nombre
        if candidata.is_dir():
            return candidata
    # Respaldo: carpetas del curso en la unidad del sistema (VM de laboratorio)
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    respaldo = [curso / "paquete" / nombre]
    if nombre == "02_Datos_Practica":
        respaldo.insert(0, curso / "datos")
    for candidata in respaldo:
        if candidata.is_dir():
            return candidata
    if obligatorio:
        raise FileNotFoundError(
            f"No se encuentra la carpeta '{nombre}'. El cuaderno se abre desde su carpeta "
            "dentro del repositorio del curso, sin moverlo de lugar."
        )
    return None


DATOS = localizar_carpeta("02_Datos_Practica")
print("Datos de práctica encontrados:", DATOS.name)
print("Subcarpetas:", sorted(p.name for p in DATOS.iterdir() if p.is_dir()))


## Paso 0: conexión con `mi_pipeline` y rama de trabajo (C0)

In [ ]:
# Conexión de mi_pipeline y de las herramientas del LAB (no se modifica)
%load_ext autoreload
%autoreload 2
import html
import json
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
import warnings

import pandas as pd
import pymupdf
import pytesseract
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)   # avisos de cambios futuros de las bibliotecas, que no cambian el resultado
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)


def localizar_raiz(max_niveles=6):
    """Carpeta del repositorio clonado que contiene mi_pipeline."""
    if os.environ.get("MI_PIPELINE_RAIZ"):
        return Path(os.environ["MI_PIPELINE_RAIZ"])
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        if (carpeta / "mi_pipeline" / "__init__.py").is_file():
            return carpeta
    clon = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso" / "trabajo" / "entorno-extraccion-documental"
    if (clon / "mi_pipeline" / "__init__.py").is_file():
        return clon
    raise FileNotFoundError("No se encuentra la carpeta mi_pipeline: la preparación de la sesión la repone.")


RAIZ = localizar_raiz()
sys.path.insert(0, str(RAIZ))
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)
CONTROLES = []
INICIO = time.perf_counter()
pytesseract.pytesseract.tesseract_cmd = (
    os.environ.get("TESSERACT_CMD")
    or shutil.which("tesseract")
    or str(Path(os.environ.get("ProgramFiles", "")) / "Tesseract-OCR" / "tesseract.exe")
)


def correr_pruebas():
    """Corre las pruebas de mi_pipeline con pytest, muestra el resumen y dice si todas pasaron."""
    corrida = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider",
         "--basetemp", tempfile.mkdtemp(prefix="pytest_s07_"), str(RAIZ / "mi_pipeline" / "tests")],
        capture_output=True, text=True, cwd=str(RAIZ), timeout=900, encoding="utf-8", errors="replace",
    )
    resumen = [linea for linea in corrida.stdout.strip().splitlines() if " passed" in linea or " failed" in linea or "error" in linea]
    print(re.sub(r" in [\d.]+s.*", "", resumen[-1]) if resumen else corrida.stdout[-500:])
    return corrida.returncode == 0


# Cómo se muestran los resultados: tablas alineadas, con el texto completo y los números como se escriben en el Perú
PORCENTAJES = {"exactitud"}
SIN_MILES = {"contexto", "nivel"}


def num(valor, decimales=2):
    """Un número con punto de miles, coma decimal y sin ceros de más: 7.728 o 10,78."""
    texto = f"{valor:,.{decimales}f}".replace(",", "§").replace(".", ",").replace("§", ".")
    return texto.rstrip("0").rstrip(",") if "," in texto else texto


def pct(valor):
    """Una proporción como porcentaje: 95,8 %."""
    return f"{valor * 100:.1f} %".replace(".", ",")


def si_no(valor):
    return "sí" if valor else "no"


def como_texto(valor, columna="", vacio="vacío"):
    """Un valor tal como se muestra en una tabla: sí o no, números en formato local y «vacío» si no hay dato."""
    if pd.api.types.is_scalar(valor) and pd.isna(valor):
        return vacio
    if hasattr(valor, "item"):
        valor = valor.item()
    if isinstance(valor, bool):
        return si_no(valor)
    if isinstance(valor, (int, float)):
        if columna in PORCENTAJES:
            return pct(valor)
        if columna in SIN_MILES:
            return str(valor)
        return num(valor, 3 if "CER" in columna else 2)
    return str(valor)


def ver(tabla, titulo, vacio="vacío"):
    """Muestra una tabla alineada y con el texto completo; la tabla original no cambia."""
    if tabla.empty:
        print(f"{titulo}. Ninguno.")
        return
    base = tabla.reset_index() if tabla.index.name else tabla
    vista = pd.DataFrame({columna: [como_texto(v, str(columna), vacio) for v in base[columna]] for columna in base.columns})
    cuerpo = (vista.to_html(index=False, border=0)
              .replace("<th>", '<th style="text-align:left;vertical-align:bottom;padding:4px 10px">')
              .replace("<td>", '<td style="text-align:left;vertical-align:top;white-space:normal;padding:4px 10px">'))
    display({"text/html": f"<p><b>{html.escape(titulo)}</b></p>{cuerpo}", "text/plain": titulo + "\n" + vista.to_string(index=False)}, raw=True)


def decir(texto):
    """Dice qué mostró esta corrida, con sus propios números."""
    display({"text/html": f'<p style="border-left:4px solid #888;padding:6px 10px"><b>En esta corrida:</b> {html.escape(texto)}</p>',
             "text/plain": "En esta corrida: " + texto}, raw=True)


# Medición contra la respuesta correcta de cada documento (la misma vara en todos los ejercicios)
NO_SE_MIDEN = {"ambiguedad", "cuadra", "paginas", "n_paginas", "monto_en_texto", "canal", "adjunto",
               "n_requisitos", "n_requisitos_cumplen", "n_requisitos_no_cumplen", "n_requisitos_observados"}
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "setiembre", "octubre", "noviembre", "diciembre"]


def llano(texto):
    """Minúsculas, sin tildes y con un solo espacio."""
    base = "".join(c for c in unicodedata.normalize("NFD", str(texto)) if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", base.lower()).strip(" .,:;")


def figura(valor, texto):
    """Dice si una fecha (en cualquiera de sus formas escritas) o un código figura en el texto."""
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
        anio, mes, dia = valor.split("-")
        meses = [MESES[int(mes) - 1]] + (["septiembre"] if mes == "09" else [])
        formas = [valor, f"{dia}/{mes}/{anio}", f"{int(dia)}/{mes}/{anio}"] + [f"{d} de {m} de {anio}" for m in meses for d in (dia, int(dia))]
        return any(forma in texto for forma in formas)
    return llano(valor) in texto


def correcto_de(id_):
    """Respuesta correcta de un documento: sus campos, con null en las fechas y códigos que no figuran en su texto."""
    datos = json.loads((DATOS / "10_Ground_Truth" / f"{id_}.json").read_text(encoding="utf-8"))
    texto = llano(datos["texto_completo"])
    correcto = {}
    for campo, valor in datos["campos"].items():
        if campo in NO_SE_MIDEN or isinstance(valor, (bool, dict, list)):
            continue
        if isinstance(valor, str) and (re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor) or campo == "numero_reclamo") and not figura(valor, texto):
            valor = None
        correcto[campo] = valor
    return correcto


def es_correcto(valor, correcto):
    """Medida estricta: el dato listo para usar (número como número, fecha AAAA-MM-DD, moneda PEN); null es correcto si no hay dato."""
    if correcto is None:
        return valor is None
    if valor is None:
        return False
    if isinstance(correcto, (int, float)):
        return isinstance(valor, (int, float)) and not isinstance(valor, bool) and abs(valor - correcto) < 0.011
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}|\d{8,20}", str(correcto)):
        return str(valor).strip() == correcto
    return llano(valor) == llano(correcto)


def aciertos(extraido, correcto):
    """(correctos, esperados, fallos) de un documento."""
    escrito = lambda v: "vacío" if v is None else repr(v)
    fallos = [f"{campo}: {escrito(extraido.get(campo))} en lugar de {escrito(valor)}" for campo, valor in correcto.items() if not es_correcto(extraido.get(campo), valor)]
    return len(correcto) - len(fallos), len(correcto), fallos


# S07: los procesos trabajadores heredan esta variable y así encuentran Tesseract
os.environ["TESSERACT_CMD"] = pytesseract.pytesseract.tesseract_cmd
os.environ.pop("MI_PIPELINE_CONFIG", None)   # la prueba de regresión usa el YAML del paquete, salvo que una celda indique otro
try:
    import pipeline_sbs  # el paquete del curso: trae el control de ingesta de S02
except ImportError:
    sys.path.insert(0, str(RAIZ / "04_pipeline_sbs"))
VENV_PRUEBA = RAIZ.parent / "venv_prueba_s07"   # el entorno limpio del E1b, junto a la copia del curso


def armar_carpeta(nombre, archivos, subcarpeta):
    """Copia documentos de práctica a una carpeta junto al repositorio (fuera de él: no entran al guardado de Git)."""
    destino = RAIZ.parent / nombre
    if not destino.is_dir() or sorted(p.name for p in destino.iterdir()) != sorted(archivos):
        shutil.rmtree(destino, ignore_errors=True)
        destino.mkdir(parents=True)
        for archivo in archivos:
            shutil.copy2(DATOS / subcarpeta / archivo, destino / archivo)
    return destino


def resumen_pytest(texto):
    """La última línea de pytest, sin los segundos: «11 passed», «6 failed, 5 passed»."""
    lineas = [linea for linea in texto.strip().splitlines() if any(p in linea for p in (" passed", " failed", " error", " skipped", " deselected"))]
    return re.sub(r" in [\d.]+s.*", "", lineas[-1]).strip("= ") if lineas else texto[-300:]


def git(*argumentos):
    """Ejecuta un comando de Git en la copia del curso y devuelve lo que imprime."""
    return subprocess.run(["git", *argumentos], cwd=RAIZ, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()

print("Carpeta del repositorio:", RAIZ.name, "| Tesseract:", pytesseract.get_tesseract_version())

In [ ]:
# ✔ Control C0: la rama de trabajo, mi_pipeline de la sesión 6 y la versión con que se empieza
rama = git("branch", "--show-current")
assert rama == "trabajo-s07", f"C0: la rama actual es '{rama}'; la preparación de la sesión crea trabajo-s07 desde trabajo-s08."
faltan = [m for m in ("orquestacion/lote.py", "orquestacion/trabajo.py", "almacenamiento/guardar.py", "__main__.py") if not (RAIZ / "mi_pipeline" / m).is_file()]
assert not faltan, f"C0: faltan módulos de la sesión 6: {faltan}. La preparación de la sesión los repone."
print("Rama:", rama, "| versión con que se empieza (git describe):", git("describe", "--tags", "--always", "--dirty"))
CONTROLES.append("C0")
print("✔ C0 superado: rama trabajo-s07 y mi_pipeline de la sesión 6 completo")

**Cómo leer esta salida**

- **Qué se ve:** La rama de hoy y la versión con que se empieza, en el formato de `git describe`.
- **Qué mirar:** La versión todavía no es `s07-v1`: esa etiqueta se pone en el E3.
- **Qué se decide:** Nada todavía: es el punto de partida que el registro va a anotar.

## E1b. La rueda: el paquete que viaja a otra máquina (C1)

◆ **Estrategias de despliegue.** Script copiado, tarea programada, rueda instalable y contenedor en un servidor: cuatro peldaños. En cada peldaño viaja algo más: el código, las dependencias y, en el contenedor, también el sistema. Lo que no está escrito en el paquete no viaja.

In [ ]:
# E1b: la rueda instalada en un entorno limpio, sin la copia del curso (C1)
import zipfile

import mi_pipeline

RUEDA = RAIZ / "dist" / f"mi_pipeline-{mi_pipeline.__version__}-py3-none-any.whl"
if not RUEDA.is_file():   # si el paso de la terminal no la dejó, se construye aquí con el mismo comando
    subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "--no-build-isolation", "-w", "dist"], cwd=RAIZ, capture_output=True)
assert mi_pipeline.__version__ == "0.7.0" and RUEDA.is_file(), "C1: falta la rueda mi_pipeline-0.7.0 en la carpeta dist (paso de la terminal)."
with zipfile.ZipFile(RUEDA) as z:
    nombres = z.namelist()
    metadatos = z.read(next(n for n in nombres if n.endswith(".dist-info/METADATA"))).decode("utf-8")
declaradas = [linea.split(": ", 1)[1] for linea in metadatos.splitlines() if linea.startswith("Requires-Dist:")]
print(f"Rueda: {RUEDA.name} | {len(nombres)} archivos | ¿lleva pruebas?: {si_no(any('/tests/' in n for n in nombres))}")
print("Dependencias declaradas:", ", ".join(declaradas))
python_limpio = VENV_PRUEBA / "Scripts" / "python.exe"
assert python_limpio.is_file(), f"C1: falta el entorno limpio {VENV_PRUEBA.name}: se crea en el paso de la terminal."
# se ejecuta desde la carpeta del entorno limpio: así Python no ve la copia del curso y usa la rueda instalada
origen = subprocess.run([str(python_limpio), "-c", "import mi_pipeline; print(mi_pipeline.__file__)"], cwd=VENV_PRUEBA,
                        capture_output=True, text=True, encoding="utf-8", errors="replace")
ayuda = subprocess.run([str(python_limpio), "-m", "mi_pipeline", "--help"], cwd=VENV_PRUEBA, capture_output=True, text=True, encoding="utf-8", errors="replace")
print("En el entorno limpio, mi_pipeline se carga desde:", " > ".join(Path(origen.stdout.strip()).parts[-3:-1]) or origen.stderr.strip()[-200:])
print("\n".join(ayuda.stdout.strip().splitlines()[:3]) or ayuda.stderr.strip()[-400:])
assert ayuda.returncode == 0 and "site-packages" in origen.stdout, "C1: en el entorno limpio, python -m mi_pipeline --help debe responder desde la rueda instalada."
CONTROLES.append("C1")
print("✔ C1 superado: la rueda se instaló en un entorno limpio y la línea de comandos responde")

**Cómo leer esta salida**

- **Qué se ve:** El nombre de la rueda, cuántos archivos lleva, las dependencias que declara y la ayuda del comando en el entorno limpio.
- **Qué mirar:** La rueda no lleva pruebas, y `mi_pipeline` se carga desde `site-packages` (la instalación), no desde la copia del curso.
- **Qué se decide:** Con la rueda y su lista de dependencias, otra persona instala el mismo pipeline sin copiar carpetas a mano.

## E2. El libro de actas y el tablero (C2)

◆ **Monitoreo básico.** Lo que no lanza error lo delatan las métricas: un PDF cortado o un formato nuevo no se caen, pero mueven las cifras. El tablero tiene cuatro cifras con su umbral: errores, revisión, confianza y caracteres por página. El registro es el libro de actas: dice qué pasó, nunca los datos de las personas.

In [ ]:
# E2: el libro de actas. Los 12 casos del registro: antes (S02, solo el DNI) y ahora (S07)
import io
import logging

from mi_pipeline.monitoreo.registro import FormateadorJson, enmascarar_dni

CASOS = [  # (caso, mensaje, campos extra, dato que no debe verse)
    ("DNI en el mensaje", "cliente 99386108", {}, "99386108"),
    ("RUC en el mensaje", "empresa RUC 20992371634", {}, "20992371634"),
    ("correo", "respuesta a daniel.rojas@ejemplo.pe", {}, "daniel.rojas"),
    ("cuenta", "abono en la cuenta 999-1278573731", {}, "1278573731"),
    ("CCI de 20 dígitos", "CCI 99912785737316006028", {}, "9991278573731600"),
    ("teléfono", "llamar al 999 000 326", {}, "999 000 326"),
    ("campo extra con texto", "documento", {"detalle": "titular 99386133"}, "99386133"),
    ("campo extra con diccionario", "documento", {"datos": {"dni": "99386133", "ruc": "20992371634"}}, "20992371634"),
    ("campo extra con lista", "documento", {"lista": ["99386133", "99838098"]}, "99838098"),
    ("salto de línea en el mensaje", "linea 1\nlinea 2 inyectada", {}, None),
    ("8 cifras que no son DNI", "monto 99400000", {}, None),
    ("fecha", "vigencia 2026-11-20", {}, None),
]


def como_en_s02(mensaje, extra):
    """Lo que escribía el formateador de S02: solo enmascaraba el DNI de los textos de primer nivel."""
    evento = {"mensaje": enmascarar_dni(mensaje), **{k: enmascarar_dni(v) if isinstance(v, str) else v for k, v in extra.items()}}
    return json.dumps(evento, ensure_ascii=False)


def como_en_s07(mensaje, extra):
    """Lo que escribe hoy el registro: la línea JSON del formateador ampliado, sin la hora ni el nivel."""
    salida = io.StringIO()
    logger = logging.getLogger("lab.s07.casos")
    logger.handlers.clear()
    logger.propagate = False
    manejador = logging.StreamHandler(salida)
    manejador.setFormatter(FormateadorJson())
    logger.addHandler(manejador)
    logger.warning(mensaje, extra=extra)
    evento = json.loads(salida.getvalue())
    return json.dumps({k: v for k, v in evento.items() if k not in ("ts", "nivel", "logger")}, ensure_ascii=False)


filas = [{"caso": caso, "antes (S02)": como_en_s02(m, e), "ahora (S07)": como_en_s07(m, e), "fuga ahora": bool(d) and d in como_en_s07(m, e)} for caso, m, e, d in CASOS]
TABLA_CASOS = pd.DataFrame(filas)
ver(TABLA_CASOS, "Los 12 casos del registro")
fugas_antes = sum(bool(d) and d in como_en_s02(m, e) for _, m, e, d in CASOS)
decir(f"antes se filtraban {fugas_antes} de los 9 datos personales; ahora, {int(TABLA_CASOS['fuga ahora'].sum())}.")
assert not TABLA_CASOS["fuga ahora"].any(), "C2: hay una fuga en el registro: se revisa mi_pipeline/monitoreo/registro.py."

**Cómo leer esta salida**

- **Qué es cada fila:** Un caso de prueba: lo que escribía el registro de S02 y lo que escribe hoy.
- **Qué mirar:** La columna «fuga ahora» debe decir «no» en todas las filas; «antes» deja ver el RUC, el correo, la cuenta, el CCI y el teléfono.
- **Qué se decide:** El registro puede compartirse con soporte técnico sin exponer datos personales.

In [ ]:
# E2: el tablero. 15 pólizas pasan por el pipeline con registro; DuckDB lee el registro y calcula las cifras (C2)
from mi_pipeline.monitoreo.metricas import alertas, calcular_metricas, tablero, umbrales_de
from mi_pipeline.monitoreo.registro import ejecutar_registrado
from mi_pipeline.orquestacion.config import cargar_config

POLIZAS = armar_carpeta("polizas_s07", [f"POL-{i:04d}.pdf" for i in range(1, 16)], "05_Polizas")
SALIDA_E2 = (SALIDAS / "e2").resolve()
shutil.rmtree(SALIDA_E2, ignore_errors=True)
CFG = cargar_config().en_carpeta(SALIDA_E2).con(modo="secuencial")
REGISTRO = SALIDA_E2 / "registro.jsonl"
resumen_e2 = ejecutar_registrado(POLIZAS, CFG, REGISTRO, avisar=print)
lineas = REGISTRO.read_text(encoding="utf-8").splitlines()
print(f"\nEl registro tiene {len(lineas)} líneas: inicio, {len(lineas) - 2} documentos y fin. La del primer documento:")
print(lineas[1])
METRICAS = calcular_metricas(REGISTRO)
AVISOS = alertas(METRICAS, umbrales_de(CFG))
print()
print(tablero(METRICAS, AVISOS))
texto_registro = REGISTRO.read_text(encoding="utf-8")
personales = [str(v) for i in range(1, 16) for c, v in correcto_de(f"POL-{i:04d}").items() if c in ("dni", "ruc", "asegurado", "beneficiario") and v]
fugas = [dato for dato in personales if dato in texto_registro]
decir(f"{METRICAS['documentos']} documentos, {pct(METRICAS['pct_revision'])} a revisión, confianza media {num(METRICAS['confianza_media'])}; "
      f"de {len(personales)} datos personales de las pólizas, {len(fugas)} llegaron al registro.")
assert METRICAS["documentos"] == 15 and not fugas, "C2: el registro debe tener los 15 documentos y ningún dato personal."
CONTROLES.append("C2")
print("✔ C2 superado: registro sin fugas en los 12 casos y en las 15 pólizas, y tablero calculado desde el registro")

**Cómo leer esta salida**

- **Qué se ve:** El avance del lote, la línea JSON de un documento y el tablero de la corrida.
- **Qué mirar:** La línea del documento trae cuántos campos quedaron vacíos y su ruta, pero ningún valor.
- **Qué se decide:** Esta corrida es la línea base del flujo: su % de revisión se anota en el YAML (`revision_linea_base`).

## E3. Golden set y versión (C3)

◆ **Versionamiento.** Una etiqueta anotada (`git tag -a s07-v1`) marca la versión aprobada. `git describe --tags --always --dirty` dice con qué versión se produjo un resultado; `-dirty` avisa cambios sin guardar. La versión va en cada corrida: el registro la anota al inicio.

In [ ]:
# E3: el golden set. Se congela la salida aprobada de 10 pólizas en mi_pipeline/tests/golden
from mi_pipeline.calidad.regresion import CARPETA_GOLDEN, congelar

BASE = congelar([f"POL-{i:04d}.pdf" for i in range(1, 11)], "poliza", cargar_config(), DATOS)
print("Carpeta:", "/".join(CARPETA_GOLDEN.parts[-3:]), "|", len(list(CARPETA_GOLDEN.glob("POL-*.json"))), "salidas aprobadas")
print("Línea base de exactitud contra la respuesta correcta:", pct(BASE["exactitud"]), "| versión con que se congeló:", BASE["version"])
una = json.loads((CARPETA_GOLDEN / "POL-0001.json").read_text(encoding="utf-8"))
print("Campos de una salida aprobada (POL-0001.json):", ", ".join(una["campos"]))

**Cómo leer esta salida**

- **Qué se ve:** Cuántas salidas aprobadas quedaron congeladas y la línea base de exactitud.
- **Qué mirar:** El golden set compara con la salida aprobada, no con la respuesta correcta: así no falla desde el primer día.
- **Qué se decide:** Congelar es aprobar: se hace después de revisar la salida.

In [ ]:
# E3: la prueba de regresión en verde, en rojo con un YAML cambiado y otra vez en verde (C3)
PRUEBA = RAIZ / "mi_pipeline" / "tests" / "test_regresion.py"
if not PRUEBA.is_file():   # si el Copy-Item del paso no se hizo, la prueba se trae aquí desde el repositorio
    shutil.copy2(RAIZ / "05_Laboratorios" / "pruebas_s07" / "test_regresion.py", PRUEBA)


def regresion(config=None):
    """Corre solo las pruebas marcadas «regresion»; con ``config``, la prueba usa ese YAML en lugar del del paquete."""
    entorno = {k: v for k, v in os.environ.items() if k != "MI_PIPELINE_CONFIG"}
    if config:
        entorno["MI_PIPELINE_CONFIG"] = str(config)
    return subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "-m", "regresion",
                           "--basetemp", tempfile.mkdtemp(prefix="pytest_s07_"), str(RAIZ / "mi_pipeline" / "tests")],
                          capture_output=True, text=True, cwd=str(RAIZ), env=entorno, encoding="utf-8", errors="replace", timeout=900)


verde = regresion()
print("1) Con el YAML del paquete:", resumen_pytest(verde.stdout))
SIN_PLN = (SALIDAS / "pipeline_sin_pln.yaml").resolve()
yaml_paquete = (RAIZ / "mi_pipeline" / "config" / "pipeline.yaml").read_text(encoding="utf-8")
SIN_PLN.write_text(yaml_paquete.replace("etapas: [control, ingesta, ocr, pln]", "etapas: [control, ingesta, ocr]"), encoding="utf-8")
rojo = regresion(SIN_PLN)
print("2) Con la etapa pln quitada del YAML:", resumen_pytest(rojo.stdout))
for linea in rojo.stdout.splitlines():
    if linea.startswith("E       AssertionError:"):
        print("     ", linea.split("AssertionError: ", 1)[1])
otra_vez = regresion()
print("3) Otra vez con el YAML del paquete:", resumen_pytest(otra_vez.stdout))
assert verde.returncode == 0 and rojo.returncode != 0 and otra_vez.returncode == 0, "C3: la regresión debe pasar, fallar con el YAML cambiado y volver a pasar."
CONTROLES.append("C3")
print("✔ C3 superado: golden set en verde, en rojo con el documento y el campo que cambió, y en verde otra vez")

**Cómo leer esta salida**

- **Qué se ve:** Tres corridas de la prueba de regresión: en verde, en rojo y en verde otra vez.
- **Qué mirar:** Cada línea roja dice el documento, el campo, el valor aprobado y el de hoy.
- **Qué se decide:** Un cambio en el YAML que mueve un campo aprobado no pasa sin que alguien lo revise.

In [ ]:
# E3: qué versión produjo el resultado. git describe, la etiqueta s07-v1 y el evento de inicio de la corrida
inicio = json.loads(REGISTRO.read_text(encoding="utf-8").splitlines()[0])
print("git describe ahora:", git("describe", "--tags", "--always", "--dirty"))
print("Etiqueta s07-v1:", "existe" if git("tag", "-l", "s07-v1") else "todavía no (se crea en la terminal al guardar este paso)")
print("Versión anotada en el evento de inicio de la corrida del E2:", inicio["version"], "| huella del YAML:", inicio["config_sha"])

**Cómo leer esta salida**

- **Qué se ve:** La versión de ahora, si existe la etiqueta y la versión que anotó la corrida del E2.
- **Qué mirar:** Si la versión termina en `-dirty`, había cambios sin guardar cuando se produjo el resultado.
- **Qué se decide:** Un informe oficial sale de una versión etiquetada y sin `-dirty` (pregunta 4 del cuestionario).

## E4. Secretos y arranque seguro (C4)

In [ ]:
# E4: los secretos se leen de variables de entorno y el registro los oculta
from mi_pipeline.config.secretos import ErrorSecreto, leer_secreto, secretos_en_config
from mi_pipeline.monitoreo.registro import abrir_registro, cerrar_registro

os.environ.setdefault("PIPELINE_CLAVE_DEMO", "valor-de-practica-s07")   # valor de práctica: fuera del cuaderno se define en la cuenta del usuario
SECRETO = leer_secreto("PIPELINE_CLAVE_DEMO")
REGISTRO_E4 = (SALIDAS / "e4" / "registro_secreto.jsonl").resolve()
REGISTRO_E4.unlink(missing_ok=True)
actas = abrir_registro(REGISTRO_E4)
actas.warning(f"conexión de prueba con {SECRETO}", extra={"evento": "prueba_secreto", "api_key": SECRETO})
cerrar_registro(actas)
print("Lo que quedó en el registro:", REGISTRO_E4.read_text(encoding="utf-8").strip())
print("Secretos escritos en un YAML de prueba:", secretos_en_config({"llm": {"backend": "servidor_compartido", "api_key": "abc123"}}))
try:
    leer_secreto("PIPELINE_VARIABLE_QUE_NO_EXISTE")
except ErrorSecreto as error:
    print("Sin la variable:", error)
assert SECRETO not in REGISTRO_E4.read_text(encoding="utf-8"), "El secreto no debe aparecer en el registro."

**Cómo leer esta salida**

- **Qué se ve:** La línea del registro con el secreto oculto, la clave detectada en un YAML de prueba y el mensaje cuando falta la variable.
- **Qué mirar:** El valor del secreto no aparece en ninguna parte del registro.
- **Qué se decide:** Los secretos viven en variables de entorno de la cuenta del usuario, fuera del repositorio.

In [ ]:
# E4: el arranque seguro. Con todo en orden no hay problemas; sin Tesseract, el comando se detiene antes de procesar (C4)
from mi_pipeline.config.secretos import es_administrador, verificar_arranque

print("Problemas con todo en orden:", verificar_arranque(CFG, POLIZAS, SALIDAS / "e4") or "ninguno")
print("¿Corre como administrador?:", si_no(es_administrador()))
version = subprocess.run([sys.executable, "-m", "mi_pipeline", "--version"], cwd=RAIZ, capture_output=True, text=True, encoding="utf-8", errors="replace")
print("python -m mi_pipeline --version ->", version.stdout.strip())
SALIDA_C4 = (SALIDAS / "e4" / "sin_tesseract").resolve()
shutil.rmtree(SALIDA_C4, ignore_errors=True)
sin_tesseract = {**os.environ, "TESSERACT_CMD": str(RAIZ.parent / "no_existe" / "tesseract.exe")}   # como en una máquina sin Tesseract
detenido = subprocess.run([sys.executable, "-m", "mi_pipeline", "--entrada", str(POLIZAS), "--salida", str(SALIDA_C4)], cwd=RAIZ, env=sin_tesseract,
                          capture_output=True, text=True, encoding="utf-8", errors="replace")
print("Código de salida:", detenido.returncode, "|", detenido.stderr.strip().splitlines()[0].replace(str(RAIZ.parent), "..."))
evento = json.loads((SALIDA_C4 / "registro.jsonl").read_text(encoding="utf-8").splitlines()[-1])
print("Evento en el registro:", evento["evento"], "| documentos procesados:", si_no((SALIDA_C4 / "resultados.duckdb").exists()))
assert detenido.returncode == 3 and evento["evento"] == "arranque_fallido" and not (SALIDA_C4 / "resultados.duckdb").exists(), \
    "C4: sin Tesseract, el comando debe detenerse con código 3 antes de procesar."
CONTROLES.append("C4")
print("✔ C4 superado: el arranque falla temprano, con un mensaje claro y su evento en el registro")

**Cómo leer esta salida**

- **Qué se ve:** Sin problemas con todo en orden; luego, el comando sin Tesseract: código 3, el mensaje y el evento.
- **Qué mirar:** No se creó la base de resultados: el lote no empezó.
- **Qué se decide:** Es mejor que el pipeline se detenga a las 03:00 con un mensaje claro que a mitad del lote.

In [ ]:
# E4: el comando completo, como lo correrá el caso: arranque, lote con registro, base, reporte y tablero
SALIDA_CLI = (SALIDAS / "cli").resolve()
shutil.rmtree(SALIDA_CLI, ignore_errors=True)
corrida = subprocess.run([sys.executable, "-m", "mi_pipeline", "--entrada", str(POLIZAS), "--salida", str(SALIDA_CLI), "--modo", "secuencial"],
                         cwd=RAIZ, capture_output=True, text=True, encoding="utf-8", errors="replace")
print("En la terminal, desde la copia del curso: python -m mi_pipeline --entrada ..\\polizas_s07 --salida s07_despliegue\\salidas\\cli --modo secuencial\n")
print("\n".join(linea for linea in corrida.stdout.strip().splitlines()[-9:] if not linea.startswith(("Reporte:", "Registro:"))))
assert corrida.returncode == 0, "El comando debe terminar sin error."

**Cómo leer esta salida**

- **Qué se ve:** Las últimas líneas del comando: estados, base y tablero.
- **Qué mirar:** El tablero al final de cada corrida, con o sin alertas.
- **Qué se decide:** Este es el comando que el líder de integración corre en el caso.

## Cierre

In [ ]:
# Cierre: los controles superados, las pruebas de mi_pipeline, la etiqueta s07-v1 y el tiempo del cuaderno
todo_pasa = correr_pruebas()
faltan = [control for control in ("C0", "C1", "C2", "C3", "C4") if control not in CONTROLES]
etiqueta = git("tag", "-l", "s07-v1")
print("Controles superados:", ", ".join(CONTROLES), "| faltan:", ", ".join(faltan) or "ninguno")
print("Etiqueta s07-v1:", "existe" if etiqueta else "falta (paso «Golden set y versión» de la guía)", "| git describe:", git("describe", "--tags", "--always", "--dirty"))
print("Minutos del cuaderno:", num((time.perf_counter() - INICIO) / 60, 1))
assert todo_pasa and not faltan and etiqueta, "Falta algún control, la etiqueta s07-v1 o alguna prueba no pasa: se revisa el paso correspondiente de la guía."

**Cómo leer esta salida**

- **Qué se ve:** Los controles, las pruebas, la etiqueta y los minutos del cuaderno.
- **Qué mirar:** «faltan: ninguno» y la etiqueta s07-v1.
- **Qué se decide:** El avance queda listo para guardar y subir.